<a href="https://colab.research.google.com/github/Murad460/Day_to_day_use_projs/blob/main/Algo_for_sleep_quality.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from dataclasses import dataclass
from statistics import stdev

@dataclass
class SleepData:
    TIB: float                 # Time in Bed (minutes)
    SOL: float                 # Sleep Onset Latency (minutes)
    WASO: float                # Wake After Sleep Onset (minutes)
    awakenings: int
    REM: float                 # REM sleep time (minutes)
    Deep: float                # Deep sleep time (minutes)
    Light: float               # Light sleep time (minutes)
    movements: int
    bedtimes: list             # List of bedtime timestamps (minutes or hours)
    waketimes: list            # List of waketime timestamps


def sleep_quality_logic(data: SleepData) -> float:
    # --- Core Safety Check ---
    if data.TIB <= 0:
        return 0.0

    # --- Core Medical Calculations ---
    TST = max(data.TIB - data.SOL - data.WASO, 1)
    SE = (TST / data.TIB) * 100

    REM_pct = data.REM / TST
    Deep_pct = data.Deep / TST
    Light_pct = data.Light / TST

    fragmentation = (data.awakenings + data.WASO) / TST
    movement = data.movements / TST

    # Schedule consistency
    bedtime_sd = stdev(data.bedtimes) if len(data.bedtimes) > 1 else 0
    waketime_sd = stdev(data.waketimes) if len(data.waketimes) > 1 else 0
    consistency = max(0, 1 - (bedtime_sd + waketime_sd) / 120)

    # --- Rule-Based Scoring ---
    efficiency_score = 1.0 if SE >= 85 else 0.7 if SE >= 75 else 0.4
    latency_score = 1.0 if data.SOL <= 15 else 0.7 if data.SOL <= 30 else 0.4

    stage_score = 1.0
    if REM_pct < 0.20 or REM_pct > 0.30:
        stage_score -= 0.3
    if Deep_pct < 0.15:
        stage_score -= 0.4
    if Light_pct > 0.60:
        stage_score -= 0.2
    stage_score = max(stage_score, 0)

    frag_score = 1.0 if fragmentation < 0.10 else 0.7 if fragmentation < 0.20 else 0.4
    move_score = 1.0 if movement < 0.05 else 0.7 if movement < 0.10 else 0.4
    cons_score = 1.0 if consistency >= 0.80 else 0.7 if consistency >= 0.60 else 0.4

    # --- Final Sleep Quality Score ---
    final_score = (
        0.30 * efficiency_score +
        0.15 * latency_score +
        0.20 * stage_score +
        0.15 * frag_score +
        0.10 * move_score +
        0.10 * cons_score
    )

    return round(final_score, 3)


In [2]:
data = SleepData(
    TIB=480,
    SOL=20,
    WASO=30,
    awakenings=2,
    REM=100,
    Deep=90,
    Light=240,
    movements=20,
    bedtimes=[23.0, 23.2, 22.9, 23.1],
    waketimes=[7.0, 7.1, 6.9, 7.0]
)

score = sleep_quality_logic(data)
print("Sleep Quality Score:", score)


Sleep Quality Score: 0.955
